# 🌊 OpinionFlow — DistilBERT Sentiment Model Training

**What this notebook does:**
1. Downloads the **Sentiment140** dataset (1.6M tweets, pre-labelled positive/negative)
2. Converts labels to a **3-class** format (Negative / Neutral / Positive)
3. Fine-tunes **`distilbert-base-uncased`** on GPU (Google Colab T4 — free tier)
4. Evaluates accuracy on a held-out test set
5. Exports the model weights as **`opinionflow_sentiment.pt`** for upload into the Streamlit app

---
**Runtime:** ~30-45 minutes on T4 GPU (free Colab tier)  
**Recommended:** `Runtime → Change runtime type → T4 GPU`

## ✅ Step 0 — GPU Check & Install Dependencies

In [ ]:
import subprocess, sys

# Verify GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected:')
    print(result.stdout[:300])
else:
    print('⚠️  No GPU found. Go to Runtime → Change runtime type → T4 GPU')

# Install/upgrade libraries
!pip install -q transformers datasets accelerate scikit-learn pandas numpy tqdm
print('\n✅ All packages installed.')

## 📥 Step 1 — Download Sentiment140 Dataset

In [ ]:
import os, zipfile, urllib.request
import pandas as pd

DATA_URL  = 'https://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip'
ZIP_PATH  = 'sentiment140.zip'
DATA_DIR  = 'sentiment140'

if not os.path.exists(DATA_DIR):
    print('Downloading Sentiment140 (80 MB)...')
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATA_DIR)
    os.remove(ZIP_PATH)
    print('✅ Download complete.')
else:
    print('✅ Dataset already present.')

# List extracted files
for f in os.listdir(DATA_DIR):
    print(' -', f)

## 🔧 Step 2 — Load & Preprocess Data

In [ ]:
import re
import numpy as np
from sklearn.model_selection import train_test_split

COLS = ['target', 'id', 'date', 'query', 'user', 'text']

# Find the training CSV (name varies across dataset versions)
train_file = None
for f in os.listdir(DATA_DIR):
    if 'training' in f.lower() and f.endswith('.csv'):
        train_file = os.path.join(DATA_DIR, f)
        break

assert train_file, 'Training CSV not found in extracted folder!'

df = pd.read_csv(
    train_file,
    encoding='latin-1',
    header=None,
    names=COLS,
)
print(f'Loaded {len(df):,} rows')
print(df['target'].value_counts())

In [ ]:
def clean_tweet(text):
    """Lightweight tweet cleaning."""
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'@\w+', '', text)              # remove @mentions
    text = re.sub(r'<[^>]+>', '', text)           # remove HTML
    text = re.sub(r'[^\w\s#.,!?\'\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:256]

# Sentiment140 uses 0 = Negative, 4 = Positive, 2 = Neutral (rare)
# Map to 3 classes: 0=Negative, 1=Neutral, 2=Positive
label_map = {0: 0, 2: 1, 4: 2}

df['label'] = df['target'].map(label_map)
df['text']  = df['text'].astype(str).apply(clean_tweet)
df = df[df['label'].notna() & (df['text'].str.len() > 5)].copy()
df['label'] = df['label'].astype(int)

print(f'After cleaning: {len(df):,} rows')
print(df['label'].value_counts())

In [ ]:
# ── Subsample for faster training (adjust SAMPLE_SIZE as needed) ────────────
# Full 1.6M tweets: ~3 hours on T4.  Subset of 200k: ~30-45 minutes.
SAMPLE_SIZE = 200_000

df_sample = df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(min(len(x), SAMPLE_SIZE // 3), random_state=42)
).reset_index(drop=True)

train_df, val_df = train_test_split(
    df_sample, test_size=0.1, random_state=42, stratify=df_sample['label']
)

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}')
print('Label distribution (train):\n', train_df['label'].value_counts())

## 🤗 Step 3 — Tokenise & Build HuggingFace Datasets

In [ ]:
import torch
from transformers import DistilBertTokenizer
from torch.utils.data import Dataset, DataLoader

MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = DistilBertTokenizer.from_pretrained(MODEL_NAME)

class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = texts.tolist()
        self.labels    = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long),
        }

BATCH_SIZE = 64

train_dataset = TweetDataset(train_df['text'], train_df['label'], tokenizer)
val_dataset   = TweetDataset(val_df['text'],   val_df['label'],   tokenizer)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Batches — Train: {len(train_loader)}  |  Val: {len(val_loader)}')

## 🏋️ Step 4 — Fine-tune DistilBERT

In [ ]:
from transformers import DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm.notebook import tqdm
import time

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_EPOCHS  = 3
LR        = 2e-5

print(f'Using device: {DEVICE}')

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: 'Negative', 1: 'Neutral', 2: 'Positive'},
    label2id={'Negative': 0, 'Neutral': 1, 'Positive': 2},
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * N_EPOCHS
scheduler  = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.06 * total_steps),
    num_training_steps=total_steps,
)

def evaluate(model, loader):
    model.eval()
    correct, total, total_loss = 0, 0, 0.0
    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(DEVICE)
            mask  = batch['attention_mask'].to(DEVICE)
            lbls  = batch['label'].to(DEVICE)
            out   = model(input_ids=ids, attention_mask=mask, labels=lbls)
            total_loss += out.loss.item()
            preds  = out.logits.argmax(dim=-1)
            correct += (preds == lbls).sum().item()
            total   += lbls.size(0)
    return total_loss / len(loader), correct / total


best_val_acc  = 0.0
history       = []

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0
    t0 = time.time()

    loop = tqdm(train_loader, desc=f'Epoch {epoch}/{N_EPOCHS}', leave=True)
    for batch in loop:
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        lbls  = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask, labels=lbls)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        preds  = out.logits.argmax(dim=-1)
        epoch_loss    += out.loss.item()
        epoch_correct += (preds == lbls).sum().item()
        epoch_total   += lbls.size(0)
        loop.set_postfix(loss=f'{out.loss.item():.4f}', acc=f'{epoch_correct/epoch_total:.4f}')

    val_loss, val_acc = evaluate(model, val_loader)
    train_acc = epoch_correct / epoch_total
    elapsed   = time.time() - t0

    history.append({
        'epoch': epoch,
        'train_loss': epoch_loss / len(train_loader),
        'train_acc':  train_acc,
        'val_loss':   val_loss,
        'val_acc':    val_acc,
    })

    print(f'\nEpoch {epoch} | Train acc: {train_acc:.4f} | Val acc: {val_acc:.4f} | Time: {elapsed/60:.1f} min')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'  ✅ New best saved (val_acc={best_val_acc:.4f})')

print(f'\n🏆 Best validation accuracy: {best_val_acc:.4f}')

## 📊 Step 5 — Training Curves & Evaluation

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ── Training curves ──────────────────────────────────────────────────────────
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist_df.epoch, hist_df.train_loss, 'o-', label='Train')
axes[0].plot(hist_df.epoch, hist_df.val_loss,   's--', label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(hist_df.epoch, hist_df.train_acc, 'o-', label='Train')
axes[1].plot(hist_df.epoch, hist_df.val_acc,   's--', label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=120)
plt.show()

# ── Confusion matrix ─────────────────────────────────────────────────────────
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        out  = model(input_ids=ids, attention_mask=mask)
        all_preds.extend(out.logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(batch['label'].numpy())

label_names = ['Negative', 'Neutral', 'Positive']
print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=label_names))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=label_names, yticklabels=label_names, cmap='Blues')
plt.title('Confusion Matrix'); plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120)
plt.show()

## 💾 Step 6 — Export Model & Save to Google Drive

In [ ]:
from google.colab import drive, files

# ── Option A: Mount Google Drive and save there (recommended) ────────────────
SAVE_TO_DRIVE = True   # Set False to just download directly

if SAVE_TO_DRIVE:
    drive.mount('/content/drive')
    DRIVE_PATH = '/content/drive/MyDrive/OpinionFlow'
    os.makedirs(DRIVE_PATH, exist_ok=True)

    # Copy model weights
    import shutil
    shutil.copy('best_model.pt', f'{DRIVE_PATH}/opinionflow_sentiment.pt')
    shutil.copy('training_curves.png', f'{DRIVE_PATH}/training_curves.png')
    shutil.copy('confusion_matrix.png', f'{DRIVE_PATH}/confusion_matrix.png')

    # Save tokenizer config alongside the model
    tokenizer.save_pretrained(f'{DRIVE_PATH}/tokenizer')

    print(f'✅ Model saved to Google Drive: {DRIVE_PATH}/opinionflow_sentiment.pt')
    print('   Download it from Google Drive and upload in the Streamlit sidebar.')

# ── Option B: Download directly to your computer ─────────────────────────────
print('\nDownloading model file...')
files.download('best_model.pt')
print('✅ Download started. Rename to opinionflow_sentiment.pt if needed.')

## 🧪 Step 7 — Quick Inference Test

Verify the saved model works correctly before uploading to the app.

In [ ]:
def predict_sentiment(texts, model, tokenizer, device):
    """Run inference on a list of strings."""
    model.eval()
    enc = tokenizer(
        texts, truncation=True, padding=True,
        max_length=128, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    probs  = torch.softmax(logits, dim=-1).cpu().numpy()
    preds  = probs.argmax(axis=1)
    labels = ['Negative', 'Neutral', 'Positive']
    return [(labels[p], round(float(probs[i, 2] - probs[i, 0]), 4)) for i, p in enumerate(preds)]


test_texts = [
    'I absolutely love this! Best news I have heard all week.',
    'This is a complete disaster. Terrible outcome for everyone.',
    'Interesting update. Not sure yet how this will play out.',
    'Cannot believe how amazing the results turned out to be!',
    'Very disappointing. Expected so much better from this team.',
]

model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))
results = predict_sentiment(test_texts, model, tokenizer, DEVICE)

print('\nInference Results:')
print('-' * 65)
for text, (label, score) in zip(test_texts, results):
    emoji = {'Positive': '😊', 'Negative': '😞', 'Neutral': '😐'}[label]
    print(f"{emoji}  [{label:8s}  score={score:+.4f}]  {text[:55]}...")
print('-' * 65)
print('\n✅ Model is ready! Upload opinionflow_sentiment.pt in the Streamlit sidebar.')

---
## 📌 Summary

| Step | What happened |
|------|---------------|
| Dataset | Sentiment140 (1.6M tweets) downloaded and cleaned |
| Sampling | ~200k tweets balanced across 3 classes |
| Model | `distilbert-base-uncased` fine-tuned for 3 epochs |
| Output | `opinionflow_sentiment.pt` — upload via Streamlit sidebar |

**Expected accuracy:** ~87–90% on 3-class sentiment classification.  
To improve: increase `SAMPLE_SIZE`, add more epochs, or use `bert-base-uncased`.